《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 변수 사이의 관계

지금까지는 변수를 하나씩 살펴보았습니다. 이 장에서는 변수 사이의 관계를 살펴보겠습니다. 한 변수의 값으로 다른 변수에 관한 정보를 얻을 수 있다면 두 변수는 관련이 있습니다. 예를 들어 키와 체중은 관련이 있습니다. 키가 큰 사람은 대체로 체중도 많이 나갑니다. 물론 완벽한 관계는 아닙니다. 키가 작고 무거운 사람도, 키가 크고 가벼운 사람도 있습니다. 그래도 체중을 추측하려면 키를 모를 때보다 알 때 더 정확하게 맞힐 수 있습니다.

이 장에서는 변수 사이의 관계를 시각화하는 몇 가지 방법과, 관계의 강도를 수치로 나타내는 상관을 소개합니다.

[이 노트북의 원문을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/v3/nb/chap07.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

## 산점도

수학을 유난히 잘하는 사람을 만나면, 그 사람의 언어 능력이 평균보다 좋을 것이라고 예상하나요, 나쁠 것이라고 예상하나요? 한편으로는 사람들이 어느 한 분야에 특화되므로 하나를 잘하면 다른 하나는 덜 잘할 수 있다고 생각할 수 있습니다. 다른 한편으로는 전반적으로 영리한 사람이라면 두 분야 모두 평균 이상일 것이라고 예상할 수 있습니다. 어느 쪽인지 알아보겠습니다.

“1980~1984년에 태어난 미국 청소년 8,984명의 삶을 추적하는” 미국 청소년종단조사 1997(NLSY97) 자료를 사용하겠습니다. 공개 데이터셋에는 대학 입학에 흔히 쓰이는 SAT와 ACT를 비롯한 여러 표준화 시험 점수가 있습니다. 수학과 언어 영역의 점수가 따로 있으므로 두 능력 사이의 관계를 살펴볼 수 있습니다.

저는 NLS Investigator로 분석에 사용할 변수들을 추린 자료를 만들었습니다. 허락을 받아 이 추출 자료를 재배포할 수 있습니다. 자료를 내려받는 방법은 이 장의 노트북에 있습니다.

In [4]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/nlsy97-extract.csv.gz")

`read_csv`로 자료를 읽고, `replace`로 결측을 나타내는 특수 코드를 `np.nan`으로 바꿀 수 있습니다.

In [5]:
missing_codes = [-1, -2, -3, -4, -5]
nlsy = pd.read_csv("nlsy97-extract.csv.gz").replace(missing_codes, np.nan)
nlsy.shape

In [6]:
nlsy.head()

`DataFrame`에는 조사 참가자 8,984명 각각에 해당하는 행이 있고, 제가 고른 변수 34개 각각에 해당하는 열이 있습니다. 열 이름만으로는 뜻을 알기 어려우므로, 사용할 열에 이해하기 쉬운 이름을 붙이겠습니다.

In [7]:
nlsy["sat_verbal"] = nlsy["R9793800"]
nlsy["sat_math"] = nlsy["R9793900"]

두 열에는 200 미만의 값이 조금 있습니다. 최저 점수가 200이므로 가능한 점수가 아닙니다. `np.nan`으로 바꾸겠습니다.

In [8]:
columns = ["sat_verbal", "sat_math"]

for column in columns:
    invalid = nlsy[column] < 200
    nlsy.loc[invalid, column] = np.nan

다음으로 `dropna`를 사용해 두 점수가 모두 유효한 행만 선택합니다.

In [9]:
nlsy_valid = nlsy.dropna(subset=columns).copy()
nlsy_valid.shape

원문에서 다루는 SAT 점수는 평균 500, 표준편차 100을 기준으로 표준화합니다. NLSY 표본의 평균과 표준편차도 이 값에 가깝습니다.

In [10]:
sat_verbal = nlsy_valid["sat_verbal"]
sat_verbal.mean(), sat_verbal.std()

In [11]:
sat_math = nlsy_valid["sat_math"]
sat_math.mean(), sat_math.std()

두 변수 사이에 관계가 있는지 **산점도**(scatter plot)로 살펴보겠습니다.

In [12]:
plt.scatter(sat_verbal, sat_math)

decorate(xlabel="SAT Verbal", ylabel="SAT Math")

`scatter`의 기본 옵션만으로도 관계의 전체적인 모양이 보입니다. 한 영역에서 좋은 점수를 받은 사람은 다른 영역에서도 대체로 좋은 점수를 받습니다.

하지만 이 그림에는 **점 겹침**(overplotting)이 심합니다. 겹쳐 있는 점이 많아 관계를 잘못 받아들이게 할 수 있다는 뜻입니다. 점의 밀도가 가장 높은 가운데가 충분히 짙게 보이지 않아, 상대적으로 극단적인 값들이 지나치게 두드러집니다. 그 결과 이상치가 실제 비중보다 눈에 띄기 쉽습니다.

표식 크기를 줄여 덜 겹치게 하면 그림을 개선할 수 있습니다.

In [13]:
plt.scatter(sat_verbal, sat_math, s=5)

decorate(xlabel="SAT Verbal", ylabel="SAT Math")

이제 표식들이 행과 열을 이루는 것이 보입니다. 점수가 10의 배수로 반올림되어 있기 때문입니다. 이 과정에서 정보가 일부 사라졌습니다.

잃어버린 정보를 되찾을 수는 없지만, 데이터에 **지터**(jitter)를 주면 산점도에서 반올림의 영향을 줄일 수 있습니다. 무작위 잡음을 더해 반올림으로 생긴 줄무늬를 흐리는 것입니다. 다음 함수는 시퀀스를 받아 평균이 0이고 표준편차가 주어진 값인 정규분포의 난수를 더합니다. 결과는 NumPy 배열입니다.

In [14]:
def jitter(seq, std=1):
    n = len(seq)
    return np.random.normal(0, std, n) + seq

점수에 표준편차 3의 지터를 주면 산점도에서 행과 열이 더 이상 보이지 않습니다.

In [15]:
sat_verbal_jittered = jitter(sat_verbal, 3)
sat_math_jittered = jitter(sat_math, 3)

In [16]:
plt.scatter(sat_verbal_jittered, sat_math_jittered, s=5)

decorate(xlabel="SAT Verbal", ylabel="SAT Math")

지터를 주면 반올림의 시각적 영향이 줄고 관계의 모양이 분명해집니다. 하지만 일반적으로 지터는 시각화에만 사용하고, 분석에는 지터를 주기 전 자료를 사용해야 합니다.

이 예에서는 표식 크기를 줄이고 지터를 주어도 여전히 점이 일부 겹칩니다. 한 가지를 더 해 보겠습니다. `alpha` 키워드로 표식을 반투명하게 만들 수 있습니다.

In [17]:
plt.scatter(sat_verbal_jittered, sat_math_jittered, s=5, alpha=0.2)

decorate(xlabel="SAT Verbal", ylabel="SAT Math")

투명도를 주면 겹친 점이 더 짙어져 점이 몰린 정도를 알아보기 쉬워집니다.

산점도는 간단하고 널리 쓰이지만 제대로 그리기는 어려울 수 있습니다. 대개 표식 크기, 투명도, 지터를 이리저리 조절하면서 변수 사이의 관계가 가장 잘 보이는 표현을 찾아야 합니다.

## 십분위 그림

산점도는 변수 사이의 관계를 전체적으로 보여 줍니다. 하지만 관계의 성격을 더 잘 드러내는 시각화도 있습니다. 그중 하나가 **십분위 그림**(decile plot)입니다.

십분위 그림을 만들려면 응답자를 언어 점수순으로 정렬하고, **십분위 집단**(deciles)이라는 열 개 집단으로 나눕니다. `qcut` 메서드로 집단을 구할 수 있습니다.

In [18]:
deciles = pd.qcut(nlsy_valid["sat_verbal"], 10, labels=False) + 1
deciles.value_counts().sort_index()

각 십분위 집단의 응답자 수는 대략 같습니다.

이제 `groupby`로 `DataFrame`을 십분위 집단별로 나눌 수 있습니다.

In [19]:
df_groupby = nlsy_valid.groupby(deciles)
df_groupby

결과는 집단들을 나타내는 `DataFrameGroupBy` 객체입니다. 여기서 `sat_math` 열을 선택할 수 있습니다.

In [20]:
series_groupby = df_groupby["sat_math"]
series_groupby

결과는 각 십분위 집단의 수학 점수를 나타내는 `SeriesGroupBy` 객체입니다. `quantile`로 각 집단의 10·50·90백분위수를 구할 수 있습니다.

In [21]:
low = series_groupby.quantile(0.1)
median = series_groupby.quantile(0.5)
high = series_groupby.quantile(0.9)

십분위 그림은 각 집단의 이 백분위수들을 보여 줍니다. 다음 그림에서 선은 중앙값이고, 음영은 10백분위수와 90백분위수 사이입니다.

In [22]:
xs = median.index
plt.fill_between(xs, low, high, alpha=0.2)
plt.plot(xs, median, label="median")

decorate(xlabel="SAT Verbal Decile", ylabel="SAT Math")

십분위 집단 번호 대신, 각 집단의 언어 점수 중앙값을 구해 x축에 놓을 수도 있습니다.

In [23]:
xs = df_groupby["sat_verbal"].median()

plt.fill_between(xs, low, high, alpha=0.2)
plt.plot(xs, median, color="C0", label="median")

decorate(xlabel="SAT Verbal", ylabel="SAT Math")

두 변수의 관계는 선형으로 보입니다. 언어 점수 중앙값이 일정하게 늘어날 때 수학 점수 중앙값도 대략 일정하게 늘어난다는 뜻입니다.

꼭 열 집단으로 나눌 필요는 없습니다. 원하는 수의 집단으로 나누고, 이 백분위수들 외의 요약 통계량을 구할 수도 있습니다.

## 상관

NLSY 참가자 중 많은 사람이 9학년 때 피바디 개인성취검사(PIAT)의 수학 영역을 치렀습니다. 그 결과가 든 열에 이해하기 쉬운 이름을 붙이겠습니다.

In [24]:
nlsy["piat_math"] = nlsy["R1318200"]
nlsy["piat_math"].describe()

점수의 분포는 다음과 같습니다.

In [25]:
from empiricaldist import Cdf

cdf_piat_math = Cdf.from_seq(nlsy["piat_math"], name="PIAT math")
cdf_piat_math.step()
decorate(ylabel="CDF")

9학년 때 PIAT를 잘 본 학생은 12학년 때 SAT 수학도 잘 볼 가능성이 높습니다. 두 시험을 모두 치른 NLSY 참가자의 점수 관계를 다음 산점도로 보여 줍니다. `thinkstats`의 `scatter` 함수는 표식 크기와 투명도를 조절하고, 선택적으로 지터를 줍니다.

In [26]:
from thinkstats import scatter

scatter(nlsy, "piat_math", "sat_math")

decorate(xlabel="PIAT Math", ylabel="SAT Math")

예상대로 PIAT를 잘 본 학생은 SAT 수학도 잘 볼 가능성이 높습니다. 수학 능력과 언어 능력이 관련 있다면 SAT 언어도 잘 볼 것으로 예상할 수 있습니다. 다음 그림은 PIAT와 SAT 언어 점수의 관계입니다.

In [27]:
scatter(nlsy, "piat_math", "sat_verbal")

decorate(xlabel="PIAT Math", ylabel="SAT Verbal")

PIAT 점수가 높은 학생은 SAT 언어 점수도 평균적으로 높습니다.

두 산점도를 비교하면 첫 그림의 점들은 더 모여 있고, 두 번째 그림의 점들은 더 퍼져 있는 듯합니다. 이는 PIAT 수학 점수가 SAT 언어보다 SAT 수학 점수를 더 잘 예측한다는 뜻으로 볼 수 있습니다.

이 관계들의 강도를 수치로 나타낼 때는 **피어슨 상관계수**(Pearson correlation coefficient)를 사용할 수 있습니다. 흔히 간단히 “상관”이라고도 합니다. 이해하기 위해 표준화부터 시작하겠습니다.

변수를 표준화하려면 다음 함수처럼 평균을 빼고 표준편차로 나눕니다.

In [28]:
def standardize(xs):
    """Standardizes a sequence of numbers.

    xs: sequence of numbers

    returns: NumPy array
    """
    return (xs - np.mean(xs)) / np.std(xs)

사용법을 보여 주려고 `piat_math`와 `sat_math`가 모두 유효한 행을 선택하겠습니다.

In [29]:
valid = nlsy.dropna(subset=["piat_math", "sat_math"])
piat_math = valid["piat_math"]
sat_math = valid["sat_math"]

PIAT 수학 점수를 표준화합니다.

In [30]:
piat_math_standard = standardize(piat_math)
np.mean(piat_math_standard), np.std(piat_math_standard)

결과는 “z점수”라고도 하는 **표준점수**(standard scores)입니다. 계산 방식 때문에 평균은 0에 가깝고 표준편차는 1에 가깝습니다.

SAT 수학 점수도 표준화하겠습니다.

In [31]:
sat_math_standard = standardize(sat_math)
np.mean(sat_math_standard), np.std(sat_math_standard)

다음 그림은 처음 100명 참가자의 표준점수 시퀀스를 보여 줍니다.

`subplot`에 `2, 1, 1`을 전달하면 Matplotlib은 그림들을 두 행과 한 열로 배치하고 첫 번째 그림을 준비합니다. 다시 `2, 1, 2`로 호출하면 두 번째 그림을 준비합니다. `axhline`은 좌표축 너비에 걸친 수평선을 그립니다.

In [32]:
plt.subplot(2, 1, 1)
plt.axhline(0, color="gray", lw=1, alpha=0.5)
plt.plot(piat_math_standard.values[:100], label="PIAT math")
decorate(ylabel="z-score", xticks=[])

plt.subplot(2, 1, 2)
plt.axhline(0, color="gray", lw=1, alpha=0.5)
plt.plot(sat_math_standard.values[:100], label="SAT math", color="C1")
decorate(ylabel="z-score", xticks=[])

두 변수는 분명히 관련이 있습니다. 한 점수가 평균보다 높으면 다른 점수도 평균보다 높은 경향이 있습니다. 관계의 강도를 수치로 나타내려고 표준점수들을 같은 위치끼리 곱하고, 곱의 평균을 구하겠습니다.

두 점수가 모두 양수면 곱도 양수여서 평균을 높이는 쪽으로 작용합니다. 둘 다 음수여도 곱은 양수이므로 평균을 높입니다. 부호가 반대면 곱이 음수여서 평균을 낮춥니다. 따라서 곱의 평균은 두 시퀀스가 비슷하게 움직이는 정도를 측정합니다.

In [33]:
np.mean(piat_math_standard * sat_math_standard)

약 0.64인 결과가 상관계수입니다. 다음과 같이 해석할 수 있습니다. 누군가의 PIAT 수학 점수가 평균보다 1표준편차 높으면, 그 사람의 SAT 수학 점수는 평균보다 0.64표준편차 높을 것으로 예상합니다.

> 번역자 주: 이 해석은 두 변수의 평균적 관계를 선형 모형으로 볼 때의 예측입니다. 상관계수 하나만으로 모든 점수에서의 실제 조건부 평균이 정해지는 것은 아닙니다.

곱하는 순서를 바꾸어도 결과는 같습니다.

In [34]:
np.mean(sat_math_standard * piat_math_standard)

상관계수는 대칭입니다. 누군가의 SAT 수학 점수가 평균보다 1표준편차 높으면, 그 사람의 PIAT 수학 점수도 평균보다 0.64표준편차 높을 것으로 예상합니다.

상관은 흔히 사용하는 통계량이므로 NumPy에 이를 계산하는 함수가 있습니다.

In [35]:
np.corrcoef(piat_math, sat_math)

결과는 변수마다 행 하나와 열 하나가 있는 **상관행렬**(correlation matrix)입니다. 왼쪽 위는 `piat_math`와 자기 자신의 상관이고, 오른쪽 아래는 `sat_math`와 자기 자신의 상관입니다. 변수가 자기 자신과 갖는 상관은 1이며 완전한 상관을 뜻합니다. 단, 표준편차가 0인 상수 변수에서는 상관계수를 정의할 수 없습니다.

오른쪽 위와 왼쪽 아래는 각각 `piat_math`와 `sat_math`, `sat_math`와 `piat_math`의 상관이며, 두 값은 같습니다.

`thinkstats`의 `corrcoef`는 `DataFrame`과 두 열 이름을 받아 두 열이 모두 유효한 행을 선택하고 상관을 계산합니다.

In [36]:
from thinkstats import corrcoef

corrcoef(nlsy, "piat_math", "sat_math")

이 함수로 `piat_math`와 `sat_verbal`의 상관을 구할 수 있습니다.

In [37]:
corrcoef(nlsy, "piat_math", "sat_verbal")

상관은 약 0.51입니다. PIAT 수학 점수가 평균보다 1표준편차 높으면 SAT 언어 점수는 평균보다 0.51표준편차 높을 것으로 예측합니다.

예상대로 PIAT 수학 점수는 SAT 언어보다 SAT 수학 점수를 더 잘 예측합니다.

## 상관의 강도

산점도를 많이 보다 보면 서로 다른 상관이 어떤 모습인지 감을 잡게 됩니다. 다음 그림은 그 감을 기르도록 서로 다른 상관을 갖게 생성한 무작위 자료의 산점도를 보여 줍니다.

In [38]:
np.random.seed(17)
xs = np.random.normal(size=300)
ys = np.random.normal(size=300)

In [39]:
from thinkstats import make_correlated_scatter

plt.figure(figsize=(10, 2.5))

for i, rho in enumerate([0, 0.3, 0.7, 0.99]):
    plt.subplot(1, 4, i + 1)
    make_correlated_scatter(xs, ys, rho)
decorate()

그리스 문자 ρ는 영어로 “rho”라고 쓰고 “로”라고 읽으며, 관례적으로 상관계수를 나타내는 기호입니다.

상관은 음수일 수도 있습니다. 다음은 여러 음의 상관을 갖는 무작위 자료의 산점도입니다.

In [40]:
plt.figure(figsize=(10, 2.5))

for i, rho in enumerate([-0.1, -0.3, -0.7, -0.99]):
    plt.subplot(1, 4, i + 1)
    make_correlated_scatter(xs, ys, rho)
decorate()

상관계수는 언제나 -1과 1 사이입니다. 두 변수가 독립이라면 상관은 0입니다. 하지만 상관이 0이라고 해서 관계가 없다는 뜻은 아닙니다.

특히 비선형 관계에서는 상관계수가 0에 가까울 수 있습니다. 다음 예들은 한 변수의 값을 알면 다른 변수를 훨씬 잘 예측할 수 있다는 점에서 분명한 관계가 있습니다. 그런데도 모두 상관계수는 0에 가깝습니다.

In [41]:
from thinkstats import make_nonlinear_scatter

plt.figure(figsize=(10, 2.5))

for i, kind in enumerate(["abs", "quadratic", "sinusoid"]):
    plt.subplot(1, 4, i + 1)
    make_nonlinear_scatter(xs, ys, kind)
decorate()

상관은 변수 사이의 선형 관계가 얼마나 강한지를 수치로 나타냅니다. 비선형 관계에서는 상관계수가 오해를 부를 수 있습니다. 상관이 0에 가깝다고 해서 관계가 없는 것은 아닙니다.

## 순위상관

NLSY는 종단조사입니다. 같은 사람들을 시간에 따라 추적한다는 뜻입니다. 지금 살펴보는 집단은 1980~1984년생입니다. SAT를 치른 사람은 대개 18세쯤인 1990년대 말에 시험을 보았을 것입니다. 따라서 2021년에 소득을 조사했을 때는 30대 후반이나 40대 초반이었습니다. 소득 자료가 든 열에도 이해하기 쉬운 이름을 붙이겠습니다.

In [42]:
nlsy["income"] = nlsy["U4949700"]
nlsy["income"].describe()

이 열은 세전 가족 총소득입니다. 응답자와 같은 가구의 다른 구성원이 모든 소득원에서 얻은 소득의 합이며, 미국 달러(USD)로 보고한 값입니다. 소득 분포는 다음과 같습니다.

In [43]:
cdf_income = Cdf.from_seq(nlsy["income"])
cdf_income.step()

decorate(xlabel="Income (USD)", ylabel="CDF")

60만 달러 근처의 계단을 눈여겨보세요. 참가자의 익명성을 보호하려고 이 기준을 넘는 소득에 상한 처리를 했습니다. 다음은 응답자의 PIAT 수학 점수와 훗날의 소득을 나타낸 산점도입니다.

> 번역자 주: 원문의 이 문장은 SAT 수학이라고 쓰여 있지만, 아래 코드와 축은 `piat_math`를 사용합니다. 코드에 맞춰 PIAT 수학으로 옮겼습니다.

In [44]:
scatter(nlsy, "piat_math", "income")

decorate(xlabel="PIAT math", ylabel="Gross Family Income (USD)")

두 변수 사이에 관계가 있어 보입니다. 상관을 구해 보겠습니다.

In [45]:
corrcoef(nlsy, "piat_math", "income")

상관은 약 0.3입니다. 15세 때 PIAT 수학 점수가 평균보다 1표준편차 높았던 사람은 40세 때 소득이 평균보다 약 0.3표준편차 높을 것으로 선형 예측할 수 있습니다. PIAT와 SAT 점수 사이의 상관만큼 강하지는 않지만, 소득에 영향을 주는 요인이 얼마나 많은지 생각하면 꽤 강한 편입니다.

사실 피어슨 상관계수는 이 관계의 강도를 작게 나타낼 수 있습니다. 앞 산점도에서 두 변수 모두 극단적인 값이 많아 보입니다. 상관계수는 평균으로부터의 편차를 곱해 계산하므로 이런 극단적인 값에 민감합니다.

더 강건한 대안은 **순위상관**(rank correlation)입니다. 표준점수 대신 점수의 순위를 사용합니다. pandas의 `rank` 메서드로 각 점수와 소득의 순위를 구할 수 있습니다.

In [46]:
valid = nlsy.dropna(subset=["piat_math", "income"])

piat_math_rank = valid["piat_math"].rank(method="first")
income_rank = valid["income"].rank(method="first")

`method="first"`를 전달하면 `rank`는 1부터 시퀀스 길이인 4,101까지 순위를 매깁니다. 값이 같으면 먼저 나타난 값에 앞 순위를 줍니다.

In [47]:
income_rank.min(), income_rank.max()

수학 점수 순위와 소득 순위의 산점도입니다.

In [48]:
plt.scatter(piat_math_rank, income_rank, s=5, alpha=0.2)

decorate(xlabel="PIAT math rank", ylabel="Income rank")

순위들의 상관을 계산합니다.

In [49]:
np.corrcoef(piat_math_rank, income_rank)[0, 1]

결과는 약 0.38로, 피어슨 상관인 0.30보다 조금 높습니다. 순위상관은 극단적인 값의 영향을 덜 받으므로 두 변수의 관계 강도를 나타내는 데 더 나을 수 있습니다.

`thinkstats`의 `rankcorr` 함수에 이 절의 코드가 정리되어 있습니다.

In [50]:
from thinkstats import rankcorr

rankcorr(nlsy, "piat_math", "income")

SciPy에도 비슷한 `spearmanr` 함수가 있습니다. 순위상관을 스피어먼 상관이라고도 부르기 때문입니다.

> 번역자 주: 앞의 `rank(method="first")`는 동점에도 서로 다른 순위를 줍니다. `rankcorr`와 `spearmanr`는 동점에 평균 순위를 주므로 앞 계산과 값이 조금 다릅니다. 기본적인 스피어먼 상관은 뒤의 두 함수가 계산하는 값입니다.

In [51]:
from scipy.stats import spearmanr

spearmanr(valid["piat_math"], valid["income"]).statistic

연습문제에서는 SAT 언어 점수와 소득 사이의 피어슨 상관과 순위상관을 구해 보겠습니다.

## 상관과 인과

변수 A와 B에서 상관이 관측되면, 표본추출의 우연한 결과일 수도 있고 대표성 없는 표본을 뽑은 결과일 수도 있으며, 모집단에 실제 상관이 있다는 뜻일 수도 있습니다.

모집단에도 상관이 있다면 대표적으로 세 가지 설명을 생각할 수 있습니다. A가 B의 원인이거나, B가 A의 원인이거나, 다른 요인들이 A와 B 모두의 원인일 수 있습니다. 이런 설명을 “인과관계”라고 합니다.

상관만으로는 이 설명들을 구별할 수 없어 어느 것이 맞는지 알 수 없습니다. 이 원칙을 흔히 “상관이 인과를 뜻하지는 않는다”라고 요약합니다. 워낙 간결한 표현이라 위키백과 문서도 따로 있습니다.

[상관이 인과를 뜻하지는 않는다](https://en.wikipedia.org/wiki/Correlation_does_not_imply_causation)

인과관계의 근거를 제시하려면 무엇을 할 수 있을까요?

1. 시간 순서를 이용합니다. A가 B보다 먼저 일어났다면 A가 B의 원인일 수는 있지만 반대 방향은 아닙니다. 사건의 순서는 인과의 방향을 추론하는 데 도움이 됩니다. 다만 다른 무언가가 A와 B 모두의 원인일 가능성은 남습니다.
2. 무작위성을 이용합니다. 참가자를 두 집단에 무작위로 배정하면 처치 외의 특성도 평균적으로 비슷해집니다. 이렇게 하면 관측된 차이가 다른 요인 때문에 생겼을 가능성을 줄일 수 있습니다.

이 생각이 **무작위 대조시험**(randomized controlled trial)의 바탕입니다. 참가자를 무작위로 두 집단 이상에 배정합니다. **처치군**(treatment group)은 새 약 같은 개입을 받고, **대조군**(control group)은 개입을 받지 않거나 효과가 알려진 다른 처치를 받습니다. 잘 설계된 무작위 대조시험은 처치의 인과효과를 검토하는 신뢰할 만한 방법이며, 근거중심의학의 토대입니다.

안타깝게도 대조시험이 불가능하거나 윤리에 어긋나는 경우가 있습니다. 대안은 **자연실험**(natural experiment)을 찾는 것입니다. 연구자가 통제하지 못하는 사정 때문에 비슷한 집단들이 서로 다른 조건에 놓이는 경우를 말합니다.

인과관계를 찾아내고 측정하는 통계학 분야를 **인과추론**(causal inference)이라고 합니다.

## 용어 정리

- **산점도**(scatter plot): 관측 하나마다 점 하나를 그려 두 변수 사이의 관계를 보여 주는 그림.
- **점 겹침**(overplotting): 많은 표식이 겹쳐 서로 다른 밀도의 구간을 구별하기 어렵고 관계를 잘못 보여줄 수 있는 상태.
- **지터**(jitter): 겹친 값들을 더 잘 보이게 하려고 그림의 데이터 지점에 더하는 무작위 잡음.
- **십분위 그림**(decile plot): 한 변수로 자료를 열 집단으로 나눈 뒤, 각 집단에서 다른 변수를 요약해 그린 그림.
- **십분위 집단**(decile): 자료를 정렬해 대략 같은 크기로 열 등분한 집단 중 하나.
- **피어슨 상관계수**(Pearson correlation coefficient): 두 변수 사이 선형 관계의 강도와 방향을 나타내는 통계량.
- **표준점수**(standard score): 평균에서 얼마나 떨어져 있는지 표준편차 단위로 나타내도록 표준화한 값.
- **상관행렬**(correlation matrix): 데이터셋의 변수 쌍마다 상관계수를 보여 주는 표.
- **순위상관**(rank correlation): 실제 값 대신 순위를 사용해 관계의 강도를 나타내는 강건한 방법.
- **무작위 대조시험**(randomized controlled trial): 서로 다른 처치를 받는 집단에 참가자를 무작위로 배정하는 실험.
- **처치군**(treatment group): 실험에서 검토하려는 개입을 받는 집단.
- **대조군**(control group): 개입을 받지 않거나 효과가 알려진 처치를 받는 집단.
- **자연실험**(natural experiment): 자연적으로 생긴 집단을 이용하는 실험. 때로 무작위 배정과 비슷한 상황을 만들 수 있습니다.
- **인과추론**(causal inference): 원인과 결과의 관계를 찾아내고 수치로 나타내는 방법들.

## 연습문제

### 연습문제 7.1

`thinkstats`의 `decile_plot` 함수에는 이 장 앞부분의 코드가 정리되어 있습니다. 다음과 같이 호출해 SAT 언어 점수와 수학 점수의 관계를 시각화할 수 있습니다.

In [52]:
from thinkstats import decile_plot

decile_plot(nlsy, "sat_verbal", "sat_math")
decorate(xlabel="SAT Verbal", ylabel="SAT Math")

PIAT 수학 점수와 소득의 십분위 그림을 만드세요. 선형 관계로 보이나요?

In [53]:
# Solution goes here

### 연습문제 7.2

SAT 수학 점수에 대한 소득의 산점도를 만드세요. 피어슨 상관과 순위상관을 구하세요. 두 값은 상당히 다른가요?

SAT 언어 점수에 대한 소득의 산점도를 만들고 두 상관을 구하세요. 수학과 언어 점수 중 무엇이 훗날의 소득을 더 잘 예측하나요?

In [54]:
# Solution goes here

In [55]:
# Solution goes here

### 연습문제 7.3

고등학교 평균학점(GPA)과 SAT 점수의 상관을 살펴보겠습니다. NLSY에서 GPA를 기록한 변수는 다음과 같습니다.

In [56]:
missing_codes = [-6, -7, -8, -9]
nlsy["gpa"] = nlsy["R9871900"].replace(missing_codes, np.nan) / 100
nlsy["gpa"].describe()

GPA의 분포는 다음과 같습니다.

In [57]:
cdf_income = Cdf.from_seq(nlsy["gpa"])
cdf_income.step()
decorate(xlabel="GPA", ylabel="CDF")

GPA와 SAT 수학 점수의 관계를 산점도로 그리고 상관계수를 구하세요. GPA와 SAT 언어 점수에 대해서도 같은 일을 하세요. 어느 SAT 점수가 GPA를 더 잘 예측하나요?

In [58]:
# Solution goes here

In [59]:
# Solution goes here

In [60]:
# Solution goes here

### 연습문제 7.4

교육 수준과 소득의 관계를 살펴보겠습니다. NLSY에는 각 응답자가 취득한 가장 높은 학위를 기록한 열이 있습니다. 값은 정수로 부호화되어 있습니다.

In [61]:
nlsy["degree"] = nlsy["Z9083900"]
nlsy["degree"].value_counts().sort_index()

다음 목록으로 정수의 뜻을 풀어낼 수 있습니다.

In [62]:
positions = [0, 1, 2, 3, 4, 5, 6, 7]
labels = [
    "None",
    "GED",
    "High school diploma",
    "Associate's degree",
    "Bachelor's degree",
    "Master's degree",
    "PhD",
    "Professional degree",
]

학력 분포를 나타내는 `Pmf`를 만듭니다.

In [63]:
from empiricaldist import Pmf

Pmf.from_seq(nlsy["degree"]).bar()

plt.xticks(positions, labels, rotation=30, ha="right")
decorate(ylabel="PMF")

`degree`에 대한 `income`의 산점도를 만드세요. 점 겹침을 줄이려고 `degree`에 지터를 주고, 표식 크기와 투명도를 조절하세요.

In [64]:
# Solution goes here

`groupby`로 응답자를 `degree`별로 묶으세요. `DataFrameGroupBy`에서 `income`을 선택하고 `quantile`로 각 집단의 중앙값, 10백분위수, 90백분위수를 구하세요. `fill_between`으로 10~90백분위수 사이를 채우고 `plot`으로 중앙값을 그리세요.

학위가 한 단계 높아질 때의 소득 프리미엄에 관해 무엇을 말할 수 있나요?

In [65]:
# Solution goes here

### 연습문제 7.5

> 번역자 주: 원문에는 7.4가 두 번 나옵니다. 마지막 문제를 7.5로 구분했습니다.

행동위험요인감시체계(BRFSS) 자료에는 약 40만 명이 직접 보고한 키와 체중이 있습니다. 자료를 내려받는 방법은 이 장의 노트북에 있습니다.

키와 체중의 관계를 나타내는 산점도를 만드세요. 반올림 때문에 생기는 행과 열을 흐리려면 지터가 필요할 수 있습니다. 표본이 크므로 점 겹침을 줄이도록 표식 크기와 투명도를 조절해야 합니다. 두 측정값 모두 이상치가 있으므로 `xlim`과 `ylim`으로 대부분의 응답자가 들어가는 구간을 확대해도 좋습니다.

자료는 다음과 같이 불러올 수 있습니다.

In [66]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/CDBRFS08.ASC.gz")

In [67]:
from thinkstats import read_brfss

brfss = read_brfss()
brfss["htm3"].describe()

In [68]:
# Solution goes here

In [69]:
# Solution goes here

키에 대한 체중의 십분위 그림을 만드세요. 선형 관계로 보이나요? 상관계수와 순위상관을 구하세요. 상당히 다른가요? 두 변수의 관계를 어느 쪽이 더 잘 나타낸다고 생각하나요?

In [70]:
# Solution goes here

In [71]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)